<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Comprehensive Quantization Training Guide</h1>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This notebook provides a comprehensive guide to quantization methods in LLaMA-Factory, covering:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Quantization Fundamentals</strong>: Understanding model quantization</li>
<li style="margin:6px 0;"><strong>AWQ Training</strong>: Activation-aware Weight Quantization</li>
<li style="margin:6px 0;"><strong>GPTQ Training</strong>: General Purpose Transformer Quantization</li>
<li style="margin:6px 0;"><strong>AQLM Training</strong>: Activation-aware Quantization for Language Models</li>
<li style="margin:6px 0;"><strong>OTFQ Training</strong>: On-the-fly Quantization</li>
<li style="margin:6px 0;"><strong>Quantization Evaluation</strong>: Performance benchmarking</li>
<li style="margin:6px 0;"><strong>Advanced Techniques</strong>: Mixed precision and optimization</li>
<li style="margin:6px 0;"><strong>Best Practices</strong>: Deployment and production considerations</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#setup-and-installation">Setup and Installation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#quantization-fundamentals">Quantization Fundamentals</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#awq-training">AWQ Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#gptq-training">GPTQ Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#aqlm-training">AQLM Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#otfq-training">OTFQ Training</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#evaluation">Evaluation</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#advanced-techniques">Advanced Techniques</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#best-practices">Best Practices</a></li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Setup and Installation</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">First, let's install the required dependencies for quantization training.</p>
</div>


In [ ]:
# Install quantization dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets accelerate bitsandbytes
%pip install autoawq optimum auto-gptq
%pip install matplotlib seaborn plotly pandas numpy
%pip install torch-fidelity  # For quantization quality assessment

# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AwqConfig, GptqConfig
from peft import PeftModel
import bitsandbytes as bnb
import json
import os
import yaml
from typing import Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Quantization Fundamentals</h2>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">What is Quantization?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">Quantization reduces model precision from 32-bit or 16-bit floating point to lower precision (8-bit, 4-bit, or 3-bit) to reduce memory usage and improve inference speed while maintaining model quality.</p>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Types of Quantization</h3>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Post-Training Quantization</strong>: Quantize after training</li>
<li style="margin:6px 0;"><strong>Quantization-Aware Training</strong>: Train with quantization</li>
<li style="margin:6px 0;"><strong>Dynamic Quantization</strong>: Quantize during inference</li>
<li style="margin:6px 0;"><strong>Static Quantization</strong>: Pre-compute quantization parameters</li>
</ol>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Quantization Precision</h3>
<table style="width:100%;border-collapse:collapse;background:#ffffff;font-size:0.96rem;margin:16px 0;border:1px solid #dde6ec;border-radius:16px;overflow:hidden;">
<thead>
<tr>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Precision</th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Bits</th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Memory Reduction</th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Quality Loss</th>
</tr>
</thead>
<tbody>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">FP32</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">32</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">0%</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">None</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">FP16</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">16</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">50%</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Minimal</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">INT8</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">8</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">75%</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Low</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">INT4</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">4</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">87.5%</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Moderate</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">INT3</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">3</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">90.6%</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">High</td>
</tr>
</tbody>
</table>
</div>
